In [ ]:
import warnings
import pickle
import os

from sklearn.model_selection import train_test_split, cross_val_predict, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, adjusted_mutual_info_score, balanced_accuracy_score, accuracy_score
from sklearn.utils import shuffle
from catboost import CatBoostClassifier
from cstarters.ml_utils import avg, load_dataset, featurize_alignment, confusion_matrix_plot, add_stratify_column, plot_feature_importances, importances_series_to_dict, stdev_accuracy

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=".*probability.*deprecated.*"
)

random_seed = 0

In [ ]:
data_dir = "../../data"
out_dir = "../../out"

In [ ]:
df = load_dataset(path = os.path.join(data_dir, "cstarters_dataset.csv"))

aa_properties = {
    'WOLS870101': [936, 884, 924, 98], 
    'WOLS870102': [350, 884], 
    'WOLS870103': [98, 96, 818, 936, 986], 
    'GRAR740102': [936, 98, 350], 
    'RADA880108': [347],
    'NEU1': [936, 924]
}

df = featurize_alignment(df, aa_properties)
df['starter_category'] = df['starter_category'].apply(lambda x: x[-2:])
ml_df_full = df.drop(columns=["starter_beta_hydroxylation"])
stratify_columns = ['starter_category', 'ncbi_organism_genus']
ml_df_full = add_stratify_column(ml_df_full, stratify_columns)
ml_df_full = ml_df_full.drop(columns=['ncbi_organism_genus'])
ml_df_train, ml_df_test = train_test_split(ml_df_full, test_size = 0.2, stratify = ml_df_full['stratify_column'], random_state = random_seed)

y_train = ml_df_train['starter_category']
X_train = ml_df_train.drop(columns=['cs_accession', 'starter_category', 'stratify_column', ])
X_train, y_train = shuffle(X_train, y_train, random_state=random_seed)

y_test = ml_df_test['starter_category']
X_test = ml_df_test.drop(columns=['cs_accession', 'starter_category', 'stratify_column'])
X_test, y_test = shuffle(X_test, y_test, random_state=random_seed)

y_full = ml_df_full['starter_category']
X_full = ml_df_full.drop(columns=['cs_accession', 'starter_category', 'stratify_column'])
X_full, y_full = shuffle(X_full, y_full, random_state=random_seed)


# Train

In [ ]:
# RF train
rf_params = dict(
    class_weight='balanced',
    ccp_alpha=0.01,
    max_depth=9,
    min_samples_leaf=3,
    max_leaf_nodes=7,
    min_samples_split=8,
    n_estimators=100,
    max_features="log2",
    random_state=random_seed
)
model = RandomForestClassifier(**rf_params)

cv = cross_val_score(model, X_train, y_train, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_train, y_train, cv=5)

confusion_matrix_plot(confusion_matrix(y_train, cv_pred), ["AR", "FA"], title="AR vs FA • RF • Train Dataset")

model_importances = RandomForestClassifier(**rf_params)
model_importances.fit(X_train, y_train)
importances = plot_feature_importances(model_importances, X_train.columns, combine_at_position=False)
print(importances)

In [ ]:
# SVM train
svm_params = dict(
    probability=True,
    C = 1.9, 
    class_weight='balanced',
    random_state=random_seed
)
model = svm.SVC(**svm_params)

cv = cross_val_score(model, X_train, y_train, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_train, y_train, cv=5)

confusion_matrix_plot(confusion_matrix(y_train, cv_pred), ["AR", "FA"], title="AR vs FA • SVM • Train Dataset")

In [ ]:
# LR train
lr_params = dict(
    solver='liblinear', 
    class_weight='balanced', 
    # penalty='l2', 
    C=0.3
)
model = LogisticRegression(**lr_params)

cv = cross_val_score(model, X_train, y_train, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_train, y_train, cv=5)

confusion_matrix_plot(confusion_matrix(y_train, cv_pred), ["AR", "FA"], title="AR vs FA • LR • Train Dataset")

In [ ]:
# catboost train
cat_params = dict(
    iterations=250,
    learning_rate=0.03,
    depth=3,
    l2_leaf_reg=3,
    loss_function="Logloss",
    auto_class_weights="Balanced",
    random_seed=random_seed,
    verbose=0
)

model = CatBoostClassifier(**cat_params)

cv = cross_val_score(model, X_train, y_train, cv=5)
print(cv, "avg:", avg(cv))

cv_pred = cross_val_predict(model, X_train, y_train, cv=5)

confusion_matrix_plot(confusion_matrix(y_train, cv_pred),["AR", "FA"], title="AR vs FA • CatBoost • Train Dataset")

model_importances = CatBoostClassifier(**cat_params)
model_importances.fit(X_train, y_train)
importances = plot_feature_importances(model_importances, X_train.columns, combine_at_position=False, top_n=20)

print(importances_series_to_dict(importances, num_features=15))


# Test

In [ ]:
# RF test
model = RandomForestClassifier(**rf_params)

model.fit(X_train, y_train)
model_pred = model.predict(X_test)

confusion_matrix_plot(confusion_matrix(y_test, model_pred), ["AR", "FA"], title="AR vs FA •  RF • Test Dataset")

plot_feature_importances(model, X_test.columns)

print(classification_report(y_test, model_pred, digits=3))
print("adjusted mutual information\t", round(adjusted_mutual_info_score(y_test, model_pred), 3))
print("balanced accuracy\t", round(balanced_accuracy_score(y_test, model_pred), 3))

In [ ]:
# LR test
model = LogisticRegression(**lr_params)

model.fit(X_train, y_train)
model_pred = model.predict(X_test)

confusion_matrix_plot(confusion_matrix(y_test, model_pred), ["AR", "FA"], title="AR vs FA •  LR • Test Dataset")

print(classification_report(y_test, model_pred, digits=3))
print("adjusted mutual information\t", round(adjusted_mutual_info_score(y_test, model_pred), 3))
print("balanced accuracy\t", round(balanced_accuracy_score(y_test, model_pred), 3))

In [ ]:
# SVM test
model = svm.SVC(**svm_params)

model.fit(X_train, y_train)
model_pred = model.predict(X_test)

confusion_matrix_plot(confusion_matrix(y_test, model_pred), ["AR", "FA"], title="AR vs FA • SVM • Test Dataset")

print(classification_report(y_test, model_pred, digits=3))
print("adjusted mutual information\t", round(adjusted_mutual_info_score(y_test, model_pred), 3))
print("balanced accuracy\t", round(balanced_accuracy_score(y_test, model_pred), 3))

In [ ]:
# catboost test
model = CatBoostClassifier(**cat_params)

model.fit(X_train, y_train)
model_pred = model.predict(X_test)

confusion_matrix_plot(confusion_matrix(y_test, model_pred), ["AR", "FA"], title="AR vs FA • CatBoost• Test Dataset", save_path=os.path.join(out_dir, "ml_aromaticity_test.png"), save_as_svg=True)

plot_feature_importances(model, X_test.columns, top_n=20)

print(classification_report(y_test, model_pred, digits=3))
print("adjusted mutual information\t", round(adjusted_mutual_info_score(y_test, model_pred), 3))
print("balanced accuracy\t", round(balanced_accuracy_score(y_test, model_pred), 3))

# Full

In [ ]:
# RF full
model = RandomForestClassifier(**rf_params)

cv = cross_val_score(model, X_full, y_full, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_full, y_full, cv=5)

confusion_matrix_plot(confusion_matrix(y_full, cv_pred), ["AR", "FA"], title="AR vs FA •  RF • Full Dataset")

In [ ]:
# SVM full
model = svm.SVC(**svm_params)

cv = cross_val_score(model, X_full, y_full, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_full, y_full, cv=5)

confusion_matrix_plot(confusion_matrix(y_full, cv_pred), ["AR", "FA"], title="AR vs FA • SVM • Full Dataset")
stdev_accuracy(model, X_full, y_full, iterations=100)

In [ ]:
# LR full
model = LogisticRegression(**lr_params)

cv = cross_val_score(model, X_full, y_full, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_full, y_full, cv=5)

confusion_matrix_plot(confusion_matrix(y_full, cv_pred), ["AR", "FA"], title="AR vs FA •  LogReg • Full Dataset")

In [ ]:
# catboost full
model = CatBoostClassifier(**cat_params)

cv = cross_val_score(model, X_full, y_full, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_full, y_full, cv=5)

confusion_matrix_plot(confusion_matrix(y_full, cv_pred), ["AR", "FA"], title="AR vs FA •  CatBoost• Full Dataset")

stdev_accuracy(model, X_full, y_full, iterations=100)

In [ ]:
model = CatBoostClassifier(**cat_params)

model.fit(X_full, y_full)

pickle.dump(model, open(os.path.join(out_dir, "ml_aromaticity.pkl"), "wb"))

In [ ]:
importance_model = CatBoostClassifier(**cat_params)
importance_model.fit(X_full, y_full)
plot_feature_importances(importance_model, X_full.columns, top_n=20)